# 5-Day Mean Reversion Rebound on SPY
## Strategy Brief
This strategy aims to capitalize on short-term mean reversion tendencies in the SPY ETF. The signal is generated when the SPY closes below its 5-day moving average for 5 consecutive days, indicating a potential rebound. The prediction is that SPY will revert back to its mean, leading to a short-term gain. Trades are entered on the sixth day, betting on a price increase, and exited after a fixed holding period or when a profit target is reached. Historical results suggest that this strategy can outperform a simple buy-and-hold approach during specific market conditions.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we set up the parameters for our trading strategy. This includes defining the lookback period for the moving average and the holding period for trades.

In [ ]:
# Configuration Parameters
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
MOVING_AVERAGE_WINDOW = 5
HOLDING_PERIOD = 5
PROFIT_TARGET = 0.02  # 2% profit target

## PHASE 2 - Data Exploration
We will download historical price data for SPY from Yahoo Finance and compute the 5-day moving average. This moving average will be used to identify potential mean reversion signals.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download('SPY', start=START_DATE, end=END_DATE)

# Compute 5-day moving average
data['5_MA'] = data['Close'].rolling(window=MOVING_AVERAGE_WINDOW).mean()

# Plot closing price and 5-day moving average
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='SPY Close')
plt.plot(data['5_MA'], label='5-Day MA', linestyle='--')
plt.title('SPY Close Price and 5-Day Moving Average')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.show()

## PHASE 3 - Strategy Engineering
We will generate the trading signal based on the condition that SPY closes below its 5-day moving average for 5 consecutive days. Positions will be entered on the sixth day, and we will hold for a fixed period or until a profit target is hit.

In [ ]:
# Generate signal
signal = (data['Close'] < data['5_MA']).rolling(window=MOVING_AVERAGE_WINDOW).sum() == MOVING_AVERAGE_WINDOW

# Create positions based on the signal
positions = signal.shift(1).replace(True, 1).replace(False, 0)

# Exit logic: fixed holding period or profit target
for i in range(len(positions)):
    if positions[i] == 1:
        for j in range(1, HOLDING_PERIOD + 1):
            if i + j < len(data) and (data['Close'].iloc[i + j] / data['Close'].iloc[i] - 1) >= PROFIT_TARGET:
                positions.iloc[i + j] = 0
                break

## PHASE 4 - Coding & Backtesting
We will backtest the strategy by calculating daily returns based on the positions and plotting the resulting equity curve.

In [ ]:
# Calculate daily returns
data['Returns'] = data['Close'].pct_change()

# Calculate strategy returns
strategy_returns = positions.shift(1) * data['Returns']

# Calculate equity curve
equity_curve = (1 + strategy_returns).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(equity_curve, label='Strategy Equity Curve')
plt.title('5-Day Mean Reversion Rebound Strategy Equity Curve')
plt.xlabel('Date')
plt.ylabel('Equity')
plt.legend()
plt.show()

## PHASE 5 - Performance Evaluation
We will evaluate the performance of the strategy using key metrics such as CAGR, Sharpe Ratio, Sortino Ratio, Calmar Ratio, and Maximum Drawdown. We will compare these metrics against a buy-and-hold strategy.

In [ ]:
# Performance metrics
cagr = (equity_curve.iloc[-1] ** (252.0 / len(equity_curve))) - 1
sharpe_ratio = np.mean(strategy_returns) / np.std(strategy_returns) * np.sqrt(252)
sortino_ratio = np.mean(strategy_returns) / np.std(strategy_returns[strategy_returns < 0]) * np.sqrt(252)
max_drawdown = (equity_curve / equity_curve.cummax() - 1).min()
calmar_ratio = cagr / abs(max_drawdown)

# Buy-and-hold metrics
buy_and_hold_returns = data['Returns'].cumsum()
buy_and_hold_cagr = (buy_and_hold_returns.iloc[-1] ** (252.0 / len(buy_and_hold_returns))) - 1

# Comparison table
performance_table = pd.DataFrame({
    'Strategy': [cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown],
    'Buy and Hold': [buy_and_hold_cagr, np.nan, np.nan, np.nan, np.nan]
}, index=['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'])

print(performance_table)

## PHASE 6 - Deploy & Monitor
We will create a function that downloads the last 60 days of SPY data, computes today's signal, and prints the position to take.

In [ ]:
def check_today_signal():
    # Download last 60 days of SPY data
    recent_data = yf.download('SPY', period='60d')
    
    # Compute 5-day moving average
    recent_data['5_MA'] = recent_data['Close'].rolling(window=MOVING_AVERAGE_WINDOW).mean()
    
    # Generate today's signal
    recent_signal = (recent_data['Close'] < recent_data['5_MA']).rolling(window=MOVING_AVERAGE_WINDOW).sum() == MOVING_AVERAGE_WINDOW
    
    # Determine today's position
    if recent_signal.iloc[-1]:
        print('Enter Position: Buy')
    else:
        print('No Position')

check_today_signal()